# Activations Manager

This notebook includes cells to perform different kinds of data manipulation with the activations

## Import libraries

In [ ]:
from activations import *

from utils import ACTIVATIONS_FOLDER, MODELS_FOLDER, LANGUAGES, SPLITS, MODEL_NAMES
from icecream import ic

device: t.device = t.device("cuda" if t.cuda.is_available() else "cpu")
device

## Model Downloader

In [ ]:
from models import load_and_save_model
from utils import MODEL_IDS

for model_name in MODEL_NAMES:
    model_path = Path(MODELS_FOLDER) / model_name
    if model_path.exists() and any(model_path.iterdir()):
        print(f"Skipping {model_name} ({MODEL_IDS[model_name]}), already downloaded")
        continue
    print(f"Downloading {model_name} ({MODEL_IDS[model_name]})")
    load_and_save_model(model_name)

## Activation Generator

In [ ]:
save_to_disk: bool = True
amount_of_batches_to_generate: int | None = None
batch_size: int = 128
# model_name: str = "aya_expanse"
languages_to_generate: list[str] = LANGUAGES
splits_to_generate: list[str] = SPLITS

debug = False
if debug:
    languages_to_generate = ["en"]
    languages_to_generate = LANGUAGES
    splits_to_generate = ["train"]
    amount_of_batches_to_generate = 2
    batch_size = 4
    print("Running in debug mode")

ic(save_to_disk, amount_of_batches_to_generate, batch_size, languages_to_generate, splits_to_generate)


assert save_to_disk, ("If generating, must also save to disk")

In [ ]:
languages_to_generate = LANGUAGES

# Aya Expanse is an 8B model, so it needs a smaller batch size than the 128 set above
# to fit alongside its activations on the GPU.
batch_size = 16

model_name: str = "aya_expanse"

activation_recorder: ActivationRecorder = ActivationRecorder(model_name)

special_cases: list[SpecialCase] = []    

activation_recorder.generate_all_activations(
    languages_to_generate,
    splits_to_generate,
    amount_of_batches_to_generate,
    save_to_disk,
    batch_size,
    special_cases=special_cases,
)

## Activation Merger

In [ ]:
model_names: list[str] = MODEL_NAMES
languages: list[str] = LANGUAGES
splits: list[str] = SPLITS

custom = True
if custom:
    model_names = ["aya_expanse"]
    print("Using custom configuration")
else:
    print("Using default configuration")

ic(model_names, languages, splits, custom)

for model_name in model_names:
    num_layers: int = ActivationRecorder(model_name).get_number_of_layers()
    for language in languages:
        for split in splits:
            try:
                for layer_num in range(num_layers):
                    print(f"Merging {model_name}/{language}/{split}")
                    merge_activation_batches(model_name, language, split, layer_num)
            except FileNotFoundError as e:
                print(f"Couldn't merge activations of {model_name}, {language}, {split} due to:\n {e}")
                print("It is possible that these activations were already merged")

## Activation Deleter

In [ ]:
# models_to_delete: list[str] = ["olmo_model"]
# languages_to_delete: list[str] = ["es"]
models_to_delete: list[str] = MODEL_NAMES
languages_to_delete: list[str] = LANGUAGES
splits_to_delete: list[str] = SPLITS

ignore_substring = "merged"
# ignore_substring = "batch"

actually_delete = False

for model_name in models_to_delete:
    for language in languages_to_delete:
        for split in splits_to_delete:
            delete_activations_file(model_name, language, split, layer_num=None, batch_id=None, ignore_substring=ignore_substring, actually_delete=actually_delete)
